# Session 5, Fairness audit: does the model serve every group?

> **Take the model from Notebook 1 and measure it group by group.** Same evaluation tool, the catch rate (recall), but now split by the patient's race.

The model was trained on **age and sex only**. It never saw race. So it can't be biased… can it?

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from the course's public notebook repo, nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/session5/data/covid_cdc.csv"

import pandas as pd

if DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload covid_cdc.csv ...")
        up = files.upload()
        df = pd.read_csv(next(iter(up)))
    except ImportError:
        df = pd.read_csv("data/covid_cdc.csv")   # local fallback (repo: data/covid_cdc.csv)

df.head()

### About this dataset (read this first)

`covid_cdc.csv` is a **teaching reconstruction** of the US CDC COVID-19 case-surveillance data, built to reproduce the real age / sex / race / hospitalisation relationships, but **not** a file of real individual patients (you'll see many identical rows). The patterns match documented CDC findings; the records are synthetic. That's fine for a fairness audit of the *method*, which is exactly what we're doing, but you'd re-run this on real records before acting on it.

## Step 1, Rebuild the model, keeping track of race

> Ask Claude: *"Fit the same model predicting `hospitalised` from `age_group` and `sex`, 70/30 stratified split, and keep each test patient's `race` so I can group by it."*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

y = df['hospitalised']
X = pd.get_dummies(df[['age_group', 'sex']], drop_first=True)
Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
    X, y, df.index, test_size=0.3, random_state=1, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)

SHOW_THR = 0.15                       # the cut-off you chose in Notebook 1
prob = clf.predict_proba(Xte)[:, 1]
pred = (prob >= SHOW_THR).astype(int)
print('Overall accuracy:', round(accuracy_score(yte, (prob >= 0.5)), 3))

## Step 2, Split the catch rate by race  🎯

Predict first: *the model never used race. Do you expect it to catch the same share of at-risk patients in every group?*

> Ask Claude: *"For each racial group, show how many patients needed care and what share of them the model flagged (the recall)."*

In [ ]:
race = df.loc[idx_te, 'race'].values
yv = yte.values

print(f"{'group':>30} | {'needed care':>11} | {'caught':>6} | {'catch rate':>10}")
for g in ['White', 'Asian', 'Black', 'Hispanic/Latino', 'American Indian/Alaska Native']:
    m = (race == g) & (yv == 1)          # patients in this group who needed care
    n = int(m.sum())
    if n == 0:
        continue
    caught = int((pred[m] == 1).sum())
    print(f'{g:>30} | {n:>11} | {caught:>6} | {caught / n:>10.1%}')

## Step 3, What you just found

The model catches a **much larger share of White patients** who needed care than of Black or Hispanic patients, even though it never used race.

> **Read the numbers carefully, the honest version is sharper than “White does best”:**
> - **Black** patients have a *higher* hospitalisation rate than White (≈13% vs ≈10%) yet are caught *less* (≈51% vs ≈70%). The group that needs care **more** is served **less**, this is the cleanest harm.
> - **Hispanic** patients are caught least (≈38%), but part of that is a genuinely *lower* base rate (≈8%), not only the proxy. Don't overclaim it.
> - **Sample sizes matter:** the White / Black / Hispanic groups are large enough to trust; the **Asian** and **American Indian/Alaska Native** groups are small (tens of patients), so treat those bars as noisy and build your argument on the large groups.

Now ask Claude *why*, then check the explanation yourself:

> Ask Claude: *"Show the hospitalisation rate by age group, and the average age of patients who needed care, split by race."*

The model leans on **age**. If Black and Hispanic patients tend to need care at younger ages, an age-based flag will miss them. Age acted as a **proxy**, and the harm landed on the groups already worse off. This is the real lesson of Obermeyer et al. (2019): an algorithm that never used race still under-served Black patients, because it used a proxy that correlated with race.

In [ ]:
# Check it: hospitalisation rate by age, and mean age of at-risk patients by race
df['age_mid'] = df['age_group'].str.extract(r'(\d+)').astype(float)
print(df.groupby('age_group')['hospitalised'].mean().round(3))
print()
print('mean age band start, patients who needed care, by race:')
print(df[df.hospitalised == 1].groupby('race')['age_mid'].mean().round(1))

## Step 4, Which fairness, and what would you do?

There is more than one way to be “fair”, and you usually **can't have them all at once**.

- **Equal catch rates**, flag the same share of at-risk patients in each group. Reasonable, but it can force you to treat similar patients differently.
- **Equal false-alarm rates**, raise the same share of false alarms per group. Also reasonable, and it can pull against equal catch rates.

> Ask Claude: *"Explain why equal catch rates and equal false-alarm rates can't both hold in general when the base rates differ between groups."* Then note which one matters more here, and why.

In [ ]:
# Which fairness definition matters more for early clinical outreach, and why?
#

## Step 5, Decide (this is the point)

You're advising the clinic's ethics board. Write what you would **do** and what you would **tell them**:

- What would you change, a per-group cut-off, add features, add human review, re-collect data?
- What's the trade-off of your choice (fixing one fairness measure can worsen another)?
- Could you explain to a single patient why they weren't flagged? *“The model said so” is not an answer.*

In [ ]:
# Your recommendation to the ethics board:
#

## Before you leave

1. How can a model that never uses race still treat racial groups unequally?
2. Name two fairness definitions and why you can't satisfy both.
3. One sentence for someone who says “just drop the sensitive feature and it's fair”. (Hint: proxies, see Obermeyer 2019 and the Apple Card.)